# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.tooling import tool          # turns a function into something the model can call
from lib.llm import LLM               # thin OpenAI wrapper, used for the judge
from lib.parsers import PydanticOutputParser   # JSON string -> Pydantic object

In [3]:
# .env lives at the repo root, two folders up from project/starter.
load_dotenv(dotenv_path="../../.env", override=True)

# Fail here, loudly, rather than three layers deep inside a tool call.
for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    assert os.getenv(key), f"{key} is missing from .env"

# Tavily needs its key handed over explicitly.
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# The OpenAI key is deliberately NOT stored in a variable. The OpenAI SDK reads
# OPENAI_API_KEY and OPENAI_BASE_URL from the environment on its own, and Chroma's
# embedding function is built on that same SDK, so both inherit the gateway.

print("environment loaded")

environment loaded


In [4]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(name="udaplay", embedding_function=embedding_fn)

print(f"collection ready — {collection.count()} documents")

collection ready — 15 documents


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> list:
    """Semantic search: Finds most relevant results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=3)
    return results["documents"][0]

#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Explanation of why the documents are or are not useful")


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]) -> dict:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini")

    prompt = (
        "Your task is to evaluate if the documents are enough to respond to the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{retrieved_docs}"
    )

    response = llm.invoke(input=prompt, response_format=EvaluationReport)

    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)

    return {"useful": report.useful, "description": report.description}

#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> list:
    """Web search: Finds results on the internet when the vector DB has no useful answer.
    args:
    - question: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - title: title of the web page
    - url: source URL
    - content: relevant excerpt from the page
    """
    tavily = TavilyClient(api_key=TAVILY_API_KEY)
    response = tavily.search(query=question)
    return response["results"]

In [8]:
# --- smoke test: retrieve_game ---
docs = retrieve_game.func("When was Pokémon Gold and Silver released?")
for d in docs:
    print(d)
    print("---")

[Game Boy Color] Pokémon Gold and Silver (1999) - Role-playing, published by Nintendo. Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
---
[Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Role-playing, published by Nintendo. Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.
---
[Nintendo 64] Super Mario 64 (1996) - Platformer, published by Nintendo. A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
---


In [9]:
# --- smoke test: evaluate_retrieval ---
# Case 1: good retrieval — Pokémon question, relevant docs
good_docs = retrieve_game.func("When was Pokémon Gold and Silver released?")
report_good = evaluate_retrieval.func(
    question="When was Pokémon Gold and Silver released?",
    retrieved_docs=good_docs
)
print("Case 1 (should be useful=True):")
print(report_good)

print()

# Case 2: bad retrieval — Mortal Kombat X not in corpus, noisy docs
bad_docs = retrieve_game.func("Was Mortal Kombat X released for PlayStation 5?")
report_bad = evaluate_retrieval.func(
    question="Was Mortal Kombat X released for PlayStation 5?",
    retrieved_docs=bad_docs
)
print("Case 2 (should be useful=False):")
print(report_bad)

Case 1 (should be useful=True):
{'useful': True, 'description': 'The retrieved documents contain relevant information about the release of Pokémon Gold and Silver, specifically stating that they were released in 1999. Although the other documents discuss different Pokémon games, they do not detract from the usefulness of the first document. Therefore, the information provided is sufficient to answer the question regarding the release date of Pokémon Gold and Silver.'}

Case 2 (should be useful=False):
{'useful': False, 'description': "The retrieved documents do not contain any information regarding Mortal Kombat X or its release on PlayStation 5. Instead, they focus on other games, specifically Marvel's Spider-Man and Gran Turismo 5, which are unrelated to the query. Therefore, these documents are not useful for answering whether Mortal Kombat X was released for PlayStation 5."}


In [10]:
# --- smoke test: game_web_search ---
from tavily import TavilyClient
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

web_results = game_web_search.func(question="Was Mortal Kombat X released for PlayStation 5?")
for r in web_results[:2]:
    print(r["title"])
    print(r["url"])
    print(r["content"][:200])
    print("---")

Mortal Kombat X PS5 Gameplay Review
https://www.youtube.com/watch?v=-JPXripEMoA
It's Mortal Kombat X for on PlayStation 5 taking a look at this entry in the long-running fighting series. So, what we're getting here is a 1080p 60 FPS. It features a large roster of characters with 
---
Mortal Kombat X - PS5 Gameplay
https://www.youtube.com/watch?v=tqsw711ZuAk
Mortal Kombat X is a 2015 fighting game. Entertainment for Microsoft Windows, PlayStation 4, and Xbox One.
---


### Agent

In [11]:
from lib.agents import Agent

INSTRUCTIONS = """You are UdaPlay, a research assistant for the video game industry.

Follow this procedure for every question about a game:

1. Call retrieve_game first, with the user's question as the query.
2. Call evaluate_retrieval, passing the original question and the documents
   you just retrieved.
3. If the evaluation returns useful=true, answer from those documents alone.
   If it returns useful=false, call game_web_search and answer from those
   results instead.

Never call game_web_search before steps 1 and 2. The internal database is
the preferred source.

When you answer:
- State the platform and year when the question is about a release.
- Distinguish the platforms a game was RELEASED for from platforms where it
  is merely playable through backwards compatibility. These are not the same
  claim, and web results often blur them.
- If you used web results, name the source URLs you relied on.
- If neither source answers the question, say so plainly. Do not guess."""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

print("agent ready with tools:", [t.name for t in agent.tools])

agent ready with tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']


In [14]:
from lib.messages import AIMessage


def tools_used(run):
    """Read back which tools actually ran, from the state the machine recorded."""
    names = []
    for msg in run.get_final_state()["messages"]:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            names.extend(call.function.name for call in msg.tool_calls)
    return names


QUERIES = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

# A separate session per query, so no query can answer from the previous one's
# conversation. Carrying context across turns is tested on purpose later.
for i, question in enumerate(QUERIES, start=1):
    run = agent.invoke(question, session_id=f"q{i}")
    state = run.get_final_state()

    print(f"Q{i}: {question}")
    print(f"tools fired : {tools_used(run)}")
    print(f"steps       : {len(run.snapshots)}")
    print(f"tokens      : {state.get('total_tokens', 0)}")
    print()
    print(state["messages"][-1].content)
    print("=" * 78)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q1: When was Pokémon Gold and Silver released?
tools fired : ['retrieve_game', 'evaluate_retrieval']
steps       : 7
tokens      : 2235

Pokémon Gold and Silver were released in 1999 for the Game Boy Color.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q2: Which one was the first 3D platformer Mario game?
tools fired : ['retrieve_game', 'evaluate_r

### (Optional) Advanced

In [13]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes